# MM-only dataset and mini EDA

Здесь мы оставляем только семейство `MM` и проверяем, достаточно ли этот датасет хорош для дальнейшего анализа цен опционов на фьючерс на IMOEX.

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
base = Path("/Users/maria/Desktop/Code/HSE/COURSEBOOK/data/imoex_options_2024_2026")
out_dir = Path("/Users/maria/Desktop/Code/HSE/COURSEBOOK/data/imoex_options_mm")
out_dir.mkdir(parents=True, exist_ok=True)

full_df = pd.read_parquet(base / "imoex_options_daily_history_2024_2026.parquet")
full_df["family"] = full_df["SECID"].astype(str).str[:2]

mm = full_df.loc[full_df["family"] == "MM"].copy()
mm["TRADEDATE"] = pd.to_datetime(mm["TRADEDATE"])
mm["expiry_date"] = pd.to_datetime(mm["expiry_date"])
mm = mm.drop(columns=["family"])
mm = mm.sort_values(["SECID", "TRADEDATE"]).reset_index(drop=True)

mm_parquet = out_dir / "imoex_options_mm_daily_history_2024_2026.parquet"
mm_csv = out_dir / "imoex_options_mm_daily_history_2024_2026.csv"
mm.to_parquet(mm_parquet, index=False)
mm.to_csv(mm_csv, index=False)

print("Saved:", mm_parquet)
print("Saved:", mm_csv)
print("Shape:", mm.shape)
print("Columns:", list(mm.columns))

In [ ]:
summary = {
    "rows": len(mm),
    "unique_secids": mm["SECID"].nunique(),
    "date_min": mm["TRADEDATE"].min().date(),
    "date_max": mm["TRADEDATE"].max().date(),
    "years": mm["TRADEDATE"].dt.year.value_counts().sort_index().to_dict(),
    "option_type": mm["option_type"].value_counts(dropna=False).to_dict(),
    "unique_strikes": int(mm["strike"].nunique()),
    "strike_min": float(mm["strike"].min()),
    "strike_max": float(mm["strike"].max()),
    "unique_expiries": int(mm["expiry_date"].nunique()),
    "expiry_min": mm["expiry_date"].min().date(),
    "expiry_max": mm["expiry_date"].max().date(),
}
summary

In [ ]:
missing = (
    mm[["OPEN", "HIGH", "LOW", "CLOSE", "SETTLEPRICE", "VALUE", "VOLUME", "OPENPOSITION", "NUMTRADES"]]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_share")
)

quality = {
    "duplicates_secid_tradedate": int(mm.duplicated(["SECID", "TRADEDATE"]).sum()),
    "zero_volume_share": float((mm["VOLUME"].fillna(0) == 0).mean()),
    "zero_numtrades_share": float((mm["NUMTRADES"].fillna(0) == 0).mean()),
}

missing, quality

In [ ]:
rows_by_secid = mm.groupby("SECID").size().sort_values(ascending=False)
strikes_per_date_expiry = mm.groupby(["TRADEDATE", "expiry_date"])["strike"].nunique()
horizon_days = (mm["expiry_date"] - mm["TRADEDATE"]).dt.days

print("Top SECID by rows:")
print(rows_by_secid.head(10))
print()
print("Strikes per (TRADEDATE, expiry_date):")
print(strikes_per_date_expiry.describe())
print()
print("Days to expiry:")
print(horizon_days.describe())

In [ ]:
price_stats = mm[["OPEN", "HIGH", "LOW", "CLOSE", "SETTLEPRICE", "VALUE", "VOLUME", "OPENPOSITION", "NUMTRADES"]].describe()
coverage_by_secid = mm.groupby("SECID")[["OPEN", "CLOSE", "VALUE", "VOLUME", "SETTLEPRICE"]].apply(lambda g: g.notna().mean())

print("Price stats:")
print(price_stats)
print()
print("Median non-missing share by SECID:")
print(coverage_by_secid.median())

## Короткий вывод

- Этот `MM`-датасет выглядит как наиболее подходящий рабочий набор под задачу.
- Главное достоинство: `SETTLEPRICE` заполнен полностью, покрытие по датам хорошее, дублей по `SECID x TRADEDATE` нет.
- Главное ограничение: trade-поля (`OPEN/HIGH/LOW/CLOSE/VALUE/VOLUME`) заметно неполные, и во многих строках оборот равен нулю.
- Для анализа именно **цены опциона** датасет подходит, если основной ценой считать `SETTLEPRICE`.
- Для анализа активной торговой ликвидности он уже хуже и требует осторожности.